# NLA compressor model selection

Use this notebook after candidate validation runs finish. It discovers completed direct-AE and PCA-AE artifacts without modifying or migrating either run family, and ranks validation results only. The final test split remains untouched until one configuration is frozen.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'pyproject.toml').is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError('Run this notebook from within IAFlowCloud.')
    
    PROJECT_ROOT = PROJECT_ROOT.parent
code_path = PROJECT_ROOT / 'Code'
if str(code_path) not in sys.path:
    sys.path.insert(0, str(code_path))

In [ ]:
from iaflow.comparison import (
    collect_compressor_results,
    smallest_qualified_compressor,
)

In [ ]:
REVISED_DIRECT_AE_DENSE_HIDDEN = {
    'Depth03': [256, 64, 16],
    'Depth04': [512, 256, 64, 16],
    'Depth05': [768, 512, 256, 64, 16],
}

discovered_results = collect_compressor_results(PROJECT_ROOT)
results = [
    result
    for result in discovered_results
    if result['model_family'] != 'Direct_AE'
    or result.get('dense_hidden') == REVISED_DIRECT_AE_DENSE_HIDDEN.get(result['depth'])
]
print(f'Completed canonical compressor runs: {len(results)}')
for result in results:
    reductions = result['fractional_error_reduction']
    print(
        f"{result['model_family']:>9s} {result['architecture']:>6s} "
        f"{result['depth']:>7s} "
        f"latent={result['latent_dim']:02d} "
        f"variance={result['variance_recovered']:.8%} "
        f"VR gain={result['variance_recovered_percentage_point_gain']:.5f} pp "
        f"MSE reduction={reductions['log10_mse']:.4%} "
        f"mean-relative-error reduction={reductions['mean_relative_error']:.4%}"
    )

In [ ]:
selected = smallest_qualified_compressor(
    results,
    target_variance_recovered=0.999,
)
if selected is None:
    print('No canonical candidate reaches 99.9% validation variance recovery yet.')
else:
    print('Smallest qualified validation model:')
    print(selected)